# Cp + Regroup - Separate Container Triangles

Two-step pipeline:

1. **Cp** - run the standard pressure-coefficient transform on the wind-tunnel body / probe H5s and write `cp.default.time_series.h5`.
2. **Regroup** - apply `cfdmod.regroup` with a `ByConnectivityGrouping` to detect each container as a connected component, then emit a new geometry + Cp timeseries where each container is its own labelled surface and triangles are reordered per container.

Default aggregation is `per_triangle` (one HDF5 column per parent triangle, reordered so each container is a contiguous block). Switch to `area_weighted_mean` in the regroup config cell if you want one aggregated value per container instead.

## 1. Imports

In [ ]:
import pathlib

import h5py
import numpy as np
from lnas import LnasFormat

# cfdmod v2 -- public API surface (everything we use is importable from `cfdmod` directly)
from cfdmod import (
    ByConnectivityGrouping,
    BySizeRoundedPerComponent,
    CpCaseConfig,
    RegroupConfig,
    load_mesh,
    mesh_from_h5,
)
from cfdmod.pressure.parameters import CpConfig

: 

## 2. Configure inputs

Set `DATA_DIR` to wherever the simulation results live. The cell auto-discovers the body H5 (`bodies.*.h5`) and probe H5 (`points.*.h5`) inside it; override the explicit paths below if your filenames differ.

All outputs land in `OUTPUT_DIR` (defaults to `<DATA_DIR>/output_regroup`).

In [ ]:
# --- File paths ----------------------------------------------------------
DATA_DIR = pathlib.Path.home() / "Documents/sim_results/results_container/SN"
OUTPUT_DIR = DATA_DIR / "output_regroup"

# Auto-discover body / probe H5s by the standard naming convention. Override
# explicitly if your filenames differ.
_bodies = sorted(DATA_DIR.glob("bodies.*.h5"))
_probes = sorted(DATA_DIR.glob("points.*.h5"))
BODY_H5 = _bodies[0] if _bodies else None
PROBE_H5 = _probes[0] if _probes else None

# Optional: a fixed-frame reference mesh (.lnas / .stl / .h5 / .xdmf). Leave
# as None to use the body H5's own embedded geometry.
REFERENCE_MESH = None

# --- Cp time window (None = use the body H5's full extent) ---------------
T_MIN = None
T_MAX = None

# --- Cp physics ----------------------------------------------------------
# Required by CpConfig. SIMUL_U_H is the simulation flow velocity used as
# the dynamic-pressure denominator; SIMUL_CHARACTERISTIC_LENGTH is the
# reference length (only used as the L/U time scale when normalize_time
# is True). FLUID_DENSITY defaults to 1 (LBM lattice units).
SIMUL_U_H = 1.0
SIMUL_CHARACTERISTIC_LENGTH = 1.0
FLUID_DENSITY = 1.0
MACROSCOPIC_TYPE = "pressure"  # 'pressure' or 'rho' (LBM density)
REFERENCE_PRESSURE = "probe"  # 'probe' (first probe point) or 'average'

# --- Regroup options -----------------------------------------------------
# Step 1: connectivity split. Triangle clusters smaller than MIN_TRIS are
# dropped (typically loose stray triangles).
MIN_TRIS = 4

# Step 2: subdivide each connected component by target container dimensions.
# Each component is split into max(1, round(extent / target)) cells per axis,
# anchored at that component's own bounding box. Standard 20ft container
# defaults: x=6.34m (length), y=2.58m (width), z=2.6m (height). A single
# container becomes one cell; a row of 5 touching containers becomes 5.
TARGET_SIZE_X = 6.34
TARGET_SIZE_Y = 2.58
TARGET_SIZE_Z = 2.6

# 'sliced'             : geometric 90-degree cuts at every cell boundary
#                        (Ce-style); output triangles never straddle two
#                        cells. Per-fragment HDF5 column inherits its parent
#                        triangle's value. Best for clean per-container
#                        rendering and downstream analysis.
# 'per_triangle'       : no slicing; one HDF5 column per parent triangle,
#                        reordered so each container is a contiguous block.
#                        Faster (no geometric cuts), but boundary triangles
#                        get assigned wholly to one cell by their centroid.
# 'area_weighted_mean' : one aggregated value per container (broadcast).
AGGREGATION = "sliced"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR : {DATA_DIR}")
print(f"BODY_H5  : {BODY_H5}")
print(f"PROBE_H5 : {PROBE_H5}")
print(f"OUTPUT   : {OUTPUT_DIR}")
assert (
    BODY_H5 is not None and BODY_H5.exists()
), f"No bodies.*.h5 found in {DATA_DIR}; set BODY_H5 explicitly above."
assert (
    PROBE_H5 is not None and PROBE_H5.exists()
), f"No points.*.h5 found in {DATA_DIR}; set PROBE_H5 explicitly above."

## 3. Inspect the mesh

Quick sanity check: triangle count and bounding box of the geometry the rest of the notebook will reason about.

In [ ]:
if REFERENCE_MESH is not None:
    mesh = load_mesh(REFERENCE_MESH)
    mesh_source = REFERENCE_MESH
else:
    mesh = mesh_from_h5(BODY_H5)
    mesh_source = BODY_H5
verts = mesh.geometry.vertices

print(f"Mesh source : {mesh_source}")
print(f"Triangles   : {len(mesh.geometry.triangles):,}")
print(f"Vertices    : {len(verts):,}")
print(
    f"Bounding box: "
    f"x=[{verts[:,0].min():.2f}, {verts[:,0].max():.2f}], "
    f"y=[{verts[:,1].min():.2f}, {verts[:,1].max():.2f}], "
    f"z=[{verts[:,2].min():.2f}, {verts[:,2].max():.2f}]"
)

## 4. Build the Cp config

Single body covering every surface in the mesh. We leave `statistics` unset (defaults to `[]`), which skips the stats step entirely - this notebook only needs the per-triangle Cp timeseries that regroup consumes downstream. To also write a `stats.h5`, pass e.g. `statistics=[BasicStatisticModel(stats="mean"), BasicStatisticModel(stats="rms")]` (note the field name is `stats`, plural).


In [ ]:
# Resolve the time window against the body H5's actual extent.
def _detect_time_range(body_h5: pathlib.Path) -> tuple[float, float]:
    with h5py.File(body_h5, "r") as f:
        keys = list(f["pressure"].keys())
    times = sorted(float(k[1:]) for k in keys)
    return float(times[0]), float(times[-1])


_detected = _detect_time_range(BODY_H5)
timestep_range = (
    _detected[0] if T_MIN is None else max(float(T_MIN), _detected[0]),
    _detected[1] if T_MAX is None else min(float(T_MAX), _detected[1]),
)
print(f"Body H5 time range : [{_detected[0]:.3f}, {_detected[1]:.3f}]")
print(f"Cropped to         : [{timestep_range[0]:.3f}, {timestep_range[1]:.3f}]")

cp_cfg = CpCaseConfig(
    pressure_coefficient={
        "default": CpConfig(
            timestep_range=timestep_range,
            simul_U_H=SIMUL_U_H,
            simul_characteristic_length=SIMUL_CHARACTERISTIC_LENGTH,
            fluid_density=FLUID_DENSITY,
            macroscopic_type=MACROSCOPIC_TYPE,
            reference_pressure=REFERENCE_PRESSURE,
        )
    }
)

## 5. End-to-end: slice + per-face buckets + remesh + Cp stream

One call to :func:`cfdmod.run_container_pipeline` runs the full in-memory recipe:

1. **Sliced regroup** -- cut triangles along axis-aligned cell boundaries.
2. **Per-face bucketing** -- group fragments by `(container_cell, axis-aligned face direction)`.
3. **Coarsen** -- run :func:`cfdmod.remesh_per_group` defaults (exact coplanar merge).
4. **Stream Cp** -- one pass over `BODY_H5` + `PROBE_H5` computing Cp inline, area-weight-averaging into per-face buckets, broadcasting onto the coarse mesh.

Nothing larger than the coarse mesh ever lands on disk: the 3.3 GB `cp.default.time_series.h5` and the 16 GB sliced/per-face intermediates of the previous pipeline are all gone.

Outputs:

- `regroup_out/geometry.per_face.remeshed.lnas` -- coarse mesh, one named surface per `(container, face)`.
- `regroup_out/cp.per_face.remeshed.{h5,xdmf}` -- per-face Cp animation on the coarse mesh.


In [ ]:
from cfdmod import run_container_pipeline

result = run_container_pipeline(
    body_h5=BODY_H5,
    probe_h5=PROBE_H5,
    output_dir=OUTPUT_DIR / "regroup_out",
    cp_config=cp_cfg.pressure_coefficient["default"],
    target_size_x=TARGET_SIZE_X,
    target_size_y=TARGET_SIZE_Y,
    target_size_z=TARGET_SIZE_Z,
    min_triangles=MIN_TRIS,
)
remeshed_lnas = result.remeshed_lnas
REMESHED_LNAS = result.remeshed_lnas  # alias for the inspect cell
REMESHED_H5 = result.remeshed_h5
REMESHED_XDMF = result.remeshed_xdmf

print(
    f"Sliced -> {result.n_fragments:,} fragments; "
    f"per-face buckets: {result.n_per_face_buckets:,}; "
    f"remesh: {result.n_fragments:,} -> {result.n_coarse_triangles:,} "
    f"({(1 - result.n_coarse_triangles / result.n_fragments) * 100:.1f}% reduction)"
)
print(
    f"Streamed {result.n_timesteps:,} timesteps; "
    f"timings (s): load={result.timings['load_mesh']:.2f} "
    f"slice={result.timings['slice']:.1f} "
    f"remesh={result.timings['remesh']:.1f} "
    f"stream_cp={result.timings['stream_cp']:.1f} "
    f"total={result.timings['total']:.1f}"
)
print(f"Wrote: {REMESHED_LNAS}")
print(f"       {REMESHED_H5} ({REMESHED_H5.stat().st_size / 1024**2:.1f} MB)")
print(f"       {REMESHED_XDMF}")

## 6. Inspect the coarse output

Per-face surface counts, total triangles, and a sampled summary. Each surface is named `<container>_<direction>` and typically holds two triangles (one rectangular face -> minimum triangulation).


In [ ]:
print(f"Coarse mesh: {remeshed_lnas.geometry.triangles.shape[0]:,} triangles")
print(f"Surfaces   : {len(remeshed_lnas.surfaces):,} (one per container face)")
sizes = np.array([idxs.size for idxs in remeshed_lnas.surfaces.values()])
print(
    f"Per-surface tri counts: min={int(sizes.min())} "
    f"median={int(np.median(sizes))} max={int(sizes.max())}"
)
print("\nFirst 5 surfaces (name, tri count, bounding box):")
tri_verts_out = remeshed_lnas.geometry.triangle_vertices
for name in sorted(remeshed_lnas.surfaces.keys())[:5]:
    idxs = remeshed_lnas.surfaces[name]
    if idxs.size == 0:
        continue
    sub = tri_verts_out[idxs]
    lo = sub.reshape(-1, 3).min(axis=0)
    hi = sub.reshape(-1, 3).max(axis=0)
    print(
        f"  {name:<24s} {idxs.size:>3d} tris  "
        f"x=[{lo[0]:7.2f}, {hi[0]:7.2f}]  "
        f"y=[{lo[1]:7.2f}, {hi[1]:7.2f}]  "
        f"z=[{lo[2]:7.2f}, {hi[2]:7.2f}]"
    )

## 7. Open in ParaView

| File | What it shows |
|---|---|
| `output_regroup/cp.default.time_series.xdmf` | The original Cp animation on the full unsegmented mesh. |
| `output_regroup/regroup_out/cp.per_face.remeshed.xdmf` | The per-face Cp animation on the coarse mesh -- one named surface per `(container, face)`. Use ParaView's **Extract Block** filter to isolate a single face (`<cell>_zp`, `<cell>_xn`, ...). |
